In [1]:
from pathlib import Path

import pandas as pd


In [2]:
def find_txt_files(training_root_path: Path) -> list[Path]:
    """Find all .txt files recursively under the training folder.

    Args:
            training_root_path: Root path for the training data directory.

    Returns:
            A sorted list of .txt file paths.
    """
    return sorted(training_root_path.rglob("*.txt"))


def parse_patient_txt_file(txt_file_path: Path) -> dict[str, str]:
    """Parse a patient text file of `Key: Value` rows.

    Args:
            txt_file_path: Path to one patient .txt file.

    Returns:
            A dictionary where keys are field names and values are raw string values.

    Raises:
            OSError: If the file cannot be read.
    """
    record: dict[str, str] = {}

    for raw_line in txt_file_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if ":" not in line:
            continue

        key, value = line.split(":", maxsplit=1)
        record[key.strip()] = value.strip()

    record["source_file"] = str(txt_file_path)
    return record


def build_training_dataframe(training_root_path: Path) -> pd.DataFrame:
    """Load all patient text files into one pandas DataFrame.

    Args:
            training_root_path: Root path that contains patient folders and .txt files.

    Returns:
            A DataFrame with one row per file and one column per text field.
    """
    records: list[dict[str, str]] = []
    for txt_file_path in find_txt_files(training_root_path):
        records.append(parse_patient_txt_file(txt_file_path))

    training_dataframe = pd.DataFrame.from_records(records)
    return convert_column_types(training_dataframe)


def convert_column_types(training_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Convert known columns to analysis-friendly dtypes.

    Args:
            training_dataframe: DataFrame created from patient text files.

    Returns:
            A DataFrame with converted numeric and boolean columns.
    """
    converted_dataframe = training_dataframe.copy()

    for numeric_column in ["Age", "TTM", "CPC"]:
        if numeric_column in converted_dataframe.columns:
            converted_dataframe[numeric_column] = pd.to_numeric(
                converted_dataframe[numeric_column],
                errors="coerce",
            )

    for bool_column in ["OHCA", "Shockable Rhythm"]:
        if bool_column in converted_dataframe.columns:
            converted_dataframe[bool_column] = converted_dataframe[bool_column].map(
                {"True": True, "False": False}
            )

    if "ROSC" in converted_dataframe.columns:
        converted_dataframe["ROSC"] = pd.to_numeric(
            converted_dataframe["ROSC"],
            errors="coerce",
        )

    return converted_dataframe

In [3]:
# If .parquet file already exists, load it directly for faster analysis.
parquet_path = Path("training_data.parquet")

if parquet_path.exists():
    training_dataframe = pd.read_parquet(parquet_path)
else:
    training_root_path = Path("icare_data") / "training"
    training_dataframe = build_training_dataframe(training_root_path)

    training_dataframe.to_parquet("training_data.parquet", index=False)

print("Loaded rows:", len(training_dataframe))
print("Columns:", list(training_dataframe.columns))
print(training_dataframe.head())

# Search for missing values
missing_values = training_dataframe.isnull().sum()
print("\nMissing values per column:")
print(missing_values)

# Numerical characteristics of quantitative variables
# min, q1, q2, miu, q3, max
print("\nNumerical summary:")
print(round(training_dataframe.describe(), 2))

# Table of count and percentange of Good and Poor outcomes
outcome_counts = training_dataframe["Outcome"].value_counts(dropna=False)
outcome_percentages = (
    training_dataframe["Outcome"].value_counts(dropna=False, normalize=True) * 100
)
outcome_summary = pd.DataFrame(
    {
        "Count": outcome_counts,
        "Percentage": outcome_percentages.round(2),
    }
)

print(outcome_summary)


Loaded rows: 607
Columns: ['Patient', 'Hospital', 'Age', 'Sex', 'ROSC', 'OHCA', 'Shockable Rhythm', 'TTM', 'Outcome', 'CPC', 'source_file']
  Patient Hospital   Age     Sex  ROSC   OHCA Shockable Rhythm   TTM Outcome  \
0    0284        A  53.0    Male   NaN   True             True  33.0    Good   
1    0286        F  85.0  Female   7.0  False            False   NaN    Good   
2    0296        A  48.0    Male   NaN   True             True  36.0    Good   
3    0299        A  45.0    Male   NaN   True             True  33.0    Good   
4    0303        D  51.0    Male  24.0   True             True  33.0    Good   

   CPC                        source_file  
0    1  icare_data\training\0284\0284.txt  
1    1  icare_data\training\0286\0286.txt  
2    1  icare_data\training\0296\0296.txt  
3    1  icare_data\training\0299\0299.txt  
4    1  icare_data\training\0303\0303.txt  

Missing values per column:
Patient               0
Hospital              0
Age                   1
Sex            

In [ ]:

def compute_group_stats(df: pd.DataFrame) -> dict:
    n = df["source_file"].nunique()
    age_mean = df["Age"].mean()
    age_std = df["Age"].std()
    female_count = (df["Sex"] == "Female").sum()
    female_pct = female_count / n * 100 if n > 0 else 0
    shockable_count = df["Shockable Rhythm"].sum()
    shockable_pct = df["Shockable Rhythm"].mean() * 100 if n > 0 else 0
    ttm33_count = (df["TTM"] == 33).sum()
    ttm33_pct = (df["TTM"] == 33).mean() * 100
    ttm36_count = (df["TTM"] == 36).sum()
    ttm36_pct = (df["TTM"] == 36).mean() * 100
    ttm_none_count = df["TTM"].isnull().sum()
    ttm_none_pct = df["TTM"].isnull().mean() * 100
    rosc_mean = df["ROSC"].mean()
    rosc_median = df["ROSC"].median()
    rosc_std = df["ROSC"].std()

    return {
        "Number of patients": n,
        "Age (mean ± SD)": f"{age_mean:.1f} ± {age_std:.1f}",
        "Female (%)": f"{int(female_count)} ({female_pct:.1f}%)",
        "Shockable Rhythm (%)": f"{int(shockable_count)} ({shockable_pct:.1f}%)",
        "TTM 33 (%)": f"{int(ttm33_count)} ({ttm33_pct:.1f}%)",
        "TTM 36 (%)": f"{int(ttm36_count)} ({ttm36_pct:.1f}%)",
        "No TTM (%)": f"{int(ttm_none_count)} ({ttm_none_pct:.1f}%)",
        "EEG start from ROSC, h (mean ± SD)": f"{rosc_mean:.1f} ± {rosc_std:.1f}",
        "EEG start from ROSC, h (median)": f"{rosc_median:.1f}",
    }

<img src="https://private-user-images.githubusercontent.com/81957605/558147883-801b0061-fe48-487a-933e-ede4a819389f.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NzM2ODkyMDgsIm5iZiI6MTc3MzY4ODkwOCwicGF0aCI6Ii84MTk1NzYwNS81NTgxNDc4ODMtODAxYjAwNjEtZmU0OC00ODdhLTkzM2UtZWRlNGE4MTkzODlmLnBuZz9YLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFLSUFWQ09EWUxTQTUzUFFLNFpBJTJGMjAyNjAzMTYlMkZ1cy1lYXN0LTElMkZzMyUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwMzE2VDE5MjE0OFomWC1BbXotRXhwaXJlcz0zMDAmWC1BbXotU2lnbmF0dXJlPTUzOWE5ZDFkNzRmZjg4MTIxNjQ5ZDY1MzczOTU4MmEzODUxNDViNjRlNjhhMTdiZjhkMWMwODNjNDZlMWIxM2MmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0In0.CglEm9ZS9ZiwxE3q9RSk7gBYwy4zGD68X-TefJBrPQI" width="600px" />

In [9]:
print(training_dataframe["Sex"])

0        Male
1      Female
2        Male
3        Male
4        Male
        ...  
602      Male
603      Male
604      Male
605      Male
606      Male
Name: Sex, Length: 607, dtype: str


In [ ]:
cpc_columns = {}
for cpc in [1, 2, 3, 4, 5]:
    group_df = training_dataframe[training_dataframe["CPC"] == cpc]
    cpc_columns[f"CPC {cpc}"] = compute_group_stats(group_df)

summary_table = pd.DataFrame(cpc_columns)
summary_table.index.name = "Characteristics"
display(summary_table)


,CPC 1,CPC 2,CPC 3,CPC 4,CPC 5
Characteristics,,,,,
Number of patients,181,44,20,9,353
Age (mean ± SD),57.5 ± 13.9,57.7 ± 13.5,64.7 ± 10.8,56.1 ± 20.1,63.4 ± 16.4
Female (%),50 (27.6%),12 (27.3%),6 (30.0%),3 (33.3%),116 (32.9%)
Shockable Rhythm (%),133 (77.3%),31 (77.5%),6 (40.0%),5 (55.6%),122 (36.0%)
TTM 33 (%),136 (75.1%),34 (77.3%),11 (55.0%),6 (66.7%),261 (73.9%)
TTM 36 (%),20 (11.0%),4 (9.1%),5 (25.0%),0 (0.0%),32 (9.1%)
No TTM (%),25 (13.8%),6 (13.6%),4 (20.0%),3 (33.3%),60 (17.0%)
"EEG start from ROSC, h (mean ± SD)",19.5 ± 18.0,18.0 ± 11.5,19.0 ± 13.1,24.2 ± 13.4,25.1 ± 19.6
"EEG start from ROSC, h (median)",16.0,15.0,14.5,18.5,20.0


In [ ]:
cpc_columns = {}

cpc_columns["CPC 1-2"] = compute_group_stats(
    training_dataframe[training_dataframe["CPC"].isin([1, 2])]
)
cpc_columns["CPC 3-5"] = compute_group_stats(
    training_dataframe[training_dataframe["CPC"].isin([3, 4, 5])]
)

summary_table = pd.DataFrame(cpc_columns)
summary_table.index.name = "Characteristics"
display(summary_table)


,CPC 1–2,CPC 3–5
Characteristics,,
Number of patients,225,382
Age (mean ± SD),57.5 ± 13.8,63.3 ± 16.3
Female (%),62 (27.6%),125 (32.7%)
Shockable Rhythm (%),164 (77.4%),133 (36.6%)
TTM 33 (%),170 (75.6%),278 (72.8%)
TTM 36 (%),24 (10.7%),37 (9.7%)
No TTM (%),31 (13.8%),67 (17.5%)
"EEG start from ROSC, h (mean ± SD)",19.3 ± 17.1,24.8 ± 19.2
"EEG start from ROSC, h (median)",15.0,20.0
